In [1]:
!pip install gurobipy

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import gurobipy as gp
from gurobipy import GRB

##3.1(b)

In [3]:
scenarios = [1, 2, 3, 4, 5]

# 시나리오별 확률
p = {
    1: 0.10,
    2: 0.30,
    3: 0.30,
    4: 0.20,
    5: 0.10,
}

# 시나리오별 수요
demand = {
    1: (15, 10, 5),
    2: (20, 15, 15),
    3: (25, 20, 25),
    4: (30, 25, 30),
    5: (10, 10, 10),
}

# 모델
m = gp.Model("RiskNeutral_SLP_with_Recourse")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")

# 목적식
revenue = gp.quicksum(
    p[s] * (1150*y[s,0] + 1525*y[s,1] + 1900*y[s,2])
    for s in scenarios
)
cost = 50*x[0] + 30*x[1] + 15*x[2] + 10*x[3]
m.setObjective(cost - revenue, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300, "cap_x1")
m.addConstr(x[1] <= 700, "cap_x2")
m.addConstr(x[2] <= 600, "cap_x3")
m.addConstr(x[3] <= 500, "cap_x4")
m.addConstr(x[1] + x[2] + x[3] <= 1600, "cap_x2_x3_x4")

for s in scenarios:
    m.addConstr(x[0] >=  6*y[s,0] +  8*y[s,1] + 10*y[s,2],
                f"recourse_x1_s{s}")
    m.addConstr(x[1] >= 20*y[s,0] + 25*y[s,1] + 28*y[s,2],
                f"recourse_x2_s{s}")
    m.addConstr(x[2] >= 12*y[s,0] + 15*y[s,1] + 18*y[s,2],
                f"recourse_x3_s{s}")
    m.addConstr(x[3] >=  8*y[s,0] + 10*y[s,1] + 14*y[s,2],
                f"recourse_x4_s{s}")

    da, db, dc = demand[s]
    m.addConstr(y[s,0] <= da, name=f"demand_y1_s{s}")
    m.addConstr(y[s,1] <= db, name=f"demand_y2_s{s}")
    m.addConstr(y[s,2] <= dc, name=f"demand_y3_s{s}")


m.params.OutputFlag = 1
m.optimize()

# 결과
if m.status == GRB.OPTIMAL:
    print(f"\n▶ Optimal Expected Profit = {m.objVal:.2f}\n")
    for i in range(4):
        print(f"  x{i+1} = {x[i].x:.2f}")
    for s in scenarios:
        print(f"  Scenario {s}: y1={y[s,0].x:.2f}, "
              f"y2={y[s,1].x:.2f}, y3={y[s,2].x:.2f}")
else:
    print("No optimal solution found.")


Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 40 rows, 19 columns and 102 nonzeros
Model fingerprint: 0x06bb4d02
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 21 rows, 19 columns, 83 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.6988095e+03   4.092262e+01   0.000000e+00      0s
       9   -2.3410000e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.341000000e+03

▶ Optimal

##3.2 (a)

In [4]:
cost_coef = [50, 30, 15, 10]
rev_coef = [1150, 1525, 1900]

# 기대수요
E_da = sum(p[s]*demand[s][0] for s in scenarios)
E_db = sum(p[s]*demand[s][1] for s in scenarios)
E_dc = sum(p[s]*demand[s][2] for s in scenarios)

# 모델
m_ev = gp.Model("EV_solution")

# 변수
x_ev = m_ev.addVars(4, lb=0.0, name="x_ev")
y_ev = m_ev.addVars(3, lb=0.0, name="y_ev")

# 목적식
cost = gp.quicksum(cost_coef[i] * x_ev[i] for i in range(4))
rev  = rev_coef[0] * y_ev[0] + rev_coef[1] * y_ev[1] + rev_coef[2] * y_ev[2]
m_ev.setObjective(cost - rev, GRB.MINIMIZE)

# 제약식
m_ev.addConstr(x_ev[0] <= 300)
m_ev.addConstr(x_ev[1] <= 700)
m_ev.addConstr(x_ev[2] <= 600)
m_ev.addConstr(x_ev[3] <= 500)
m_ev.addConstr(x_ev[1] + x_ev[2] + x_ev[3] <= 1600)

m_ev.addConstr( x_ev[0] >=  6*y_ev[0] +  8*y_ev[1] + 10*y_ev[2] )
m_ev.addConstr( x_ev[1] >= 20*y_ev[0] + 25*y_ev[1] + 28*y_ev[2] )
m_ev.addConstr( x_ev[2] >= 12*y_ev[0] + 15*y_ev[1] + 18*y_ev[2] )
m_ev.addConstr( x_ev[3] >=  8*y_ev[0] + 10*y_ev[1] + 14*y_ev[2] )

m_ev.addConstr( y_ev[0] <= E_da )
m_ev.addConstr( y_ev[1] <= E_db )
m_ev.addConstr( y_ev[2] <= E_dc )

m_ev.Params.OutputFlag = 0
m_ev.optimize()

print("=== EV solution ===")
print(f"Optimal EV profit = {m_ev.objVal:.2f}")
for i in range(4):
    print(f" x{i+1} = {x_ev[i].x:.2f}")
print(f" y  = {[y_ev[j].x for j in range(3)]}")
print()


=== EV solution ===
Optimal EV profit = -3233.00
 x1 = 244.28
 x2 = 700.00
 x3 = 443.40
 x4 = 334.60
 y  = [0.0, 6.16, 19.5]



## 3.2 (b)

In [5]:
#모델
m_ees = gp.Model("EES_solution")

# 변수
x_ees = m_ees.addVars(4, lb=0.0, name="x_ees")
y_ees = m_ees.addVars(3, lb=0.0, name="y_ees")

# 목적식
cost = gp.quicksum(cost_coef[i] * x_ees[i] for i in range(4))
rev  = rev_coef[0] * y_ees[0] + rev_coef[1] * y_ees[1] + rev_coef[2] * y_ees[2]
m_ees.setObjective(cost - rev, GRB.MINIMIZE)

# 제약식
m_ees.addConstr(x_ees[0] <= 300)
m_ees.addConstr(x_ees[1] <= 700)
m_ees.addConstr(x_ees[2] <= 600)
m_ees.addConstr(x_ees[3] <= 500)
m_ees.addConstr(x_ees[1] + x_ees[2] + x_ees[3] <= 1600)

m_ees.addConstr( x_ees[0] >=  6*y_ees[0] +  8*y_ees[1] + 10*y_ees[2] )
m_ees.addConstr( x_ees[1] >= 20*y_ees[0] + 25*y_ees[1] + 28*y_ees[2] )
m_ees.addConstr( x_ees[2] >= 12*y_ees[0] + 15*y_ees[1] + 18*y_ees[2] )
m_ees.addConstr( x_ees[3] >=  8*y_ees[0] + 10*y_ees[1] + 14*y_ees[2] )

m_ees.addConstr( y_ees[0] <= demand[1][0] )
m_ees.addConstr( y_ees[1] <= demand[1][1] )
m_ees.addConstr( y_ees[2] <= demand[1][2] )
m_ees.addConstr( y_ees[0] <= demand[2][0] )
m_ees.addConstr( y_ees[1] <= demand[2][1] )
m_ees.addConstr( y_ees[2] <= demand[2][2] )
m_ees.addConstr( y_ees[0] <= demand[3][0] )
m_ees.addConstr( y_ees[1] <= demand[3][1] )
m_ees.addConstr( y_ees[2] <= demand[3][2] )
m_ees.addConstr( y_ees[0] <= demand[4][0] )
m_ees.addConstr( y_ees[1] <= demand[4][1] )
m_ees.addConstr( y_ees[2] <= demand[4][2] )
m_ees.addConstr( y_ees[0] <= demand[5][0] )
m_ees.addConstr( y_ees[1] <= demand[5][1] )
m_ees.addConstr( y_ees[2] <= demand[5][2] )

m_ees.Params.OutputFlag = 0
m_ees.optimize()

print("=== EES solution ===")
print(f"Optimal profit = {m_ees.objVal:.2f}")
for i in range(4):
    print(f" x{i+1} = {x_ees[i].x:.2f}")
print(f" y  = {[y_ees[j].x for j in range(3)]}")
print()


=== EES solution ===
Optimal profit = -1250.00
 x1 = 130.00
 x2 = 390.00
 x3 = 240.00
 x4 = 170.00
 y  = [0.0, 10.0, 5.0]



## 3.2 (c)

In [6]:
for s in scenarios:
    da, db, dc = demand[s]
    m_sa = gp.Model(f"SA_s{s}")

    #변수
    x_sa = m_sa.addVars(4, lb=0.0, name="x_sa")
    y_sa = m_sa.addVars(3, lb=0.0, name="y_sa")
    # 목적식
    cost = gp.quicksum(cost_coef[i] * x_sa[i] for i in range(4))
    rev  = rev_coef[0] * y_sa[0] + rev_coef[1] * y_sa[1] + rev_coef[2] * y_sa[2]
    m_sa.setObjective(cost - rev, GRB.MINIMIZE)
    # 제약식
    m_sa.addConstr(x_sa[0] <= 300)
    m_sa.addConstr(x_sa[1] <= 700)
    m_sa.addConstr(x_sa[2] <= 600)
    m_sa.addConstr(x_sa[3] <= 500)
    m_sa.addConstr(x_sa[1] + x_sa[2] + x_sa[3] <= 1600)
    # 투입 ≥ 사용
    m_sa.addConstr( x_sa[0] >=  6*y_sa[0] +  8*y_sa[1] + 10*y_sa[2] )
    m_sa.addConstr( x_sa[1] >= 20*y_sa[0] + 25*y_sa[1] + 28*y_sa[2] )
    m_sa.addConstr( x_sa[2] >= 12*y_sa[0] + 15*y_sa[1] + 18*y_sa[2] )
    m_sa.addConstr( x_sa[3] >=  8*y_sa[0] + 10*y_sa[1] + 14*y_sa[2] )
    # 수요 한계
    m_sa.addConstr( y_sa[0] <= da )
    m_sa.addConstr( y_sa[1] <= db )
    m_sa.addConstr( y_sa[2] <= dc )

    m_sa.Params.OutputFlag = 0
    m_sa.optimize()

    print(f"Scenario {s}: profit = {m_sa.objVal:.2f}, "
          f"x = {[x_sa[i].x for i in range(4)]}, "
          f"y = {[y_sa[j].x for j in range(3)]}")

Scenario 1: profit = -1250.00, x = [130.0, 390.0, 240.0, 170.0], y = [0.0, 10.0, 5.0]
Scenario 2: profit = -2810.00, x = [239.6, 700.0, 438.0, 322.0], y = [0.0, 11.2, 15.0]
Scenario 3: profit = -3750.00, x = [250.0, 700.0, 450.0, 350.0], y = [0.0, 0.0, 25.0]
Scenario 4: profit = -3750.00, x = [250.0, 700.0, 450.00000000000006, 350.0], y = [0.0, 0.0, 25.0]
Scenario 5: profit = -2000.00, x = [180.0, 530.0, 330.0, 240.0], y = [0.0, 10.0, 10.0]


## 3.2 (d)

In [7]:
# EVPI
EVPI= -2341 - (p[1]*(-1250)+p[2]*(-2810)+p[3]*(-3750)+p[4]*(-3750)+p[5]*(-2000))
print("EVPI =", EVPI )

# VVS
m = gp.Model("RP")

# 변수
x = [244.28, 700, 443.4, 334.6]
y = m.addVars(scenarios, 3, lb=0.0, name="y")

# 목적식
revenue = gp.quicksum(
    p[s] * (1150*y[s,0] + 1525*y[s,1] + 1900*y[s,2])
    for s in scenarios
)
cost = 50*x[0] + 30*x[1] + 15*x[2] + 10*x[3]
m.setObjective(cost - revenue, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300, "cap_x1")
m.addConstr(x[1] <= 700, "cap_x2")
m.addConstr(x[2] <= 600, "cap_x3")
m.addConstr(x[3] <= 500, "cap_x4")
m.addConstr(x[1] + x[2] + x[3] <= 1600, "cap_x2_x3_x4")

for s in scenarios:
    m.addConstr(x[0] >=  6*y[s,0] +  8*y[s,1] + 10*y[s,2],
                f"recourse_x1_s{s}")
    m.addConstr(x[1] >= 20*y[s,0] + 25*y[s,1] + 28*y[s,2],
                f"recourse_x2_s{s}")
    m.addConstr(x[2] >= 12*y[s,0] + 15*y[s,1] + 18*y[s,2],
                f"recourse_x3_s{s}")
    m.addConstr(x[3] >=  8*y[s,0] + 10*y[s,1] + 14*y[s,2],
                f"recourse_x4_s{s}")

    da, db, dc = demand[s]
    m.addConstr(y[s,0] <= da, name=f"demand_y1_s{s}")
    m.addConstr(y[s,1] <= db, name=f"demand_y2_s{s}")
    m.addConstr(y[s,2] <= dc, name=f"demand_y3_s{s}")


m.params.OutputFlag = 1
m.optimize()

# 결과
VVS = -2341 - m.objVal
print("VVS =", VVS)

EVPI = 702.0
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 40 rows, 15 columns and 75 nonzeros
Model fingerprint: 0x842da314
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+02, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 7e+02]
Presolve removed 26 rows and 3 columns
Presolve time: 0.00s
Presolved: 14 rows, 12 columns, 42 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.1952667e+03   7.729250e+01   0.000000e+00      0s
      11   -2.2875000e+03   0.000000e+00   0.000000e+00      0s

Solved in 11 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.287500000e+03
VVS = -53.5


##3.3

In [8]:
# 파라미터
eta = 0       # z_RP
lam = 1000        # λ
M   = 1e6         # big‐M

m_ep = gp.Model("EP_Model")

# 변수
x = m_ep.addVars(4, lb=0.0, name="x")
y = m_ep.addVars(scenarios, 3, lb=0.0, name="y")
γ = m_ep.addVars(scenarios, vtype=GRB.BINARY, name="gamma")

# 목적식
cost_term   = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0]
          + rev_coef[1]*y[s,1]
          + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term   = lam * gp.quicksum(p[s] * γ[s] for s in scenarios)

m_ep.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m_ep.addConstr(x[0] <= 300)
m_ep.addConstr(x[1] <= 700)
m_ep.addConstr(x[2] <= 600)
m_ep.addConstr(x[3] <= 500)
m_ep.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m_ep.addConstr(x[0] >=  6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m_ep.addConstr(x[1] >= 20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m_ep.addConstr(x[2] >= 12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m_ep.addConstr(x[3] >=  8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m_ep.addConstr(y[s,0] <= da)
    m_ep.addConstr(y[s,1] <= db)
    m_ep.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m_ep.addConstr(
        - cost_term
        +        rev_s
        + M * γ[s]
        >= -eta,
        name=f"EP_s{s}"
    )

# 결과
m_ep.Params.OutputFlag = 1
m_ep.optimize()

if m_ep.status == GRB.OPTIMAL:
    print(f"\n▶ EP Model (η={eta}, λ={lam}) Optimal Obj = {m_ep.objVal:.2f}\n")
    print("x =", [x[i].x for i in range(4)])
    for s in scenarios:
        print(f"y^{s} =", [y[s,k].x for k in range(3)], "  γ^s =", int(γ[s].x))
else:
    print("No optimal solution found.")

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 24 columns and 142 nonzeros
Model fingerprint: 0x23ed0068
Variable types: 19 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+06]
  Objective range  [1e+01, 6e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 2e+03]
Found heuristic solution: objective 0.0000000
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 26 rows, 24 columns, 123 nonzeros
Variable types: 19 continuous, 5 integer (5 binary)

Root relaxation: objective -2.340640e+03, 10 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 -2340.6400    0    1    

In [9]:
# 파라미터
eta = -2341       # z_RP
lam = 1000        # λ
M   = 1e6         # big‐M

m_ep = gp.Model("EP_Model")

# 변수
x = m_ep.addVars(4, lb=0.0, name="x")
y = m_ep.addVars(scenarios, 3, lb=0.0, name="y")
γ = m_ep.addVars(scenarios, vtype=GRB.BINARY, name="gamma")

# 목적식
cost_term   = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0]
          + rev_coef[1]*y[s,1]
          + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term   = lam * gp.quicksum(p[s] * γ[s] for s in scenarios)

m_ep.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m_ep.addConstr(x[0] <= 300)
m_ep.addConstr(x[1] <= 700)
m_ep.addConstr(x[2] <= 600)
m_ep.addConstr(x[3] <= 500)
m_ep.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m_ep.addConstr(x[0] >=  6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m_ep.addConstr(x[1] >= 20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m_ep.addConstr(x[2] >= 12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m_ep.addConstr(x[3] >=  8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m_ep.addConstr(y[s,0] <= da)
    m_ep.addConstr(y[s,1] <= db)
    m_ep.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m_ep.addConstr(
        - cost_term
        +        rev_s
        + M * γ[s]
        >= -eta,
        name=f"EP_s{s}"
    )

# 결과
m_ep.Params.OutputFlag = 1
m_ep.optimize()

if m_ep.status == GRB.OPTIMAL:
    print(f"\n▶ EP Model (η={eta}, λ={lam}) Optimal Obj = {m_ep.objVal:.2f}\n")
    print("x =", [x[i].x for i in range(4)])
    for s in scenarios:
        print(f"y^{s} =", [y[s,k].x for k in range(3)], "  γ^s =", int(γ[s].x))
else:
    print("No optimal solution found.")

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 24 columns and 142 nonzeros
Model fingerprint: 0x02993b23
Variable types: 19 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+06]
  Objective range  [1e+01, 6e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 2e+03]
Found heuristic solution: objective 1000.0000000
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 26 rows, 24 columns, 123 nonzeros
Variable types: 19 continuous, 5 integer (5 binary)

Root relaxation: objective -2.334137e+03, 11 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 -2334.1373    0    2 

In [10]:
# 파라미터
eta = -2341       # z_RP
lam = 0        # λ
M   = 1e6         # big‐M

m_ep = gp.Model("EP_Model")

# 변수
x = m_ep.addVars(4, lb=0.0, name="x")
y = m_ep.addVars(scenarios, 3, lb=0.0, name="y")
γ = m_ep.addVars(scenarios, vtype=GRB.BINARY, name="gamma")

# 목적식
cost_term   = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0]
          + rev_coef[1]*y[s,1]
          + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term   = lam * gp.quicksum(p[s] * γ[s] for s in scenarios)

m_ep.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m_ep.addConstr(x[0] <= 300)
m_ep.addConstr(x[1] <= 700)
m_ep.addConstr(x[2] <= 600)
m_ep.addConstr(x[3] <= 500)
m_ep.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m_ep.addConstr(x[0] >=  6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m_ep.addConstr(x[1] >= 20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m_ep.addConstr(x[2] >= 12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m_ep.addConstr(x[3] >=  8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m_ep.addConstr(y[s,0] <= da)
    m_ep.addConstr(y[s,1] <= db)
    m_ep.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m_ep.addConstr(
        - cost_term
        +        rev_s
        + M * γ[s]
        >= -eta,
        name=f"EP_s{s}"
    )

# 결과
m_ep.Params.OutputFlag = 1
m_ep.optimize()

if m_ep.status == GRB.OPTIMAL:
    print(f"\n▶ EP Model (η={eta}, λ={lam}) Optimal Obj = {m_ep.objVal:.2f}\n")
    print("x =", [x[i].x for i in range(4)])
    for s in scenarios:
        print(f"y^{s} =", [y[s,k].x for k in range(3)], "  γ^s =", int(γ[s].x))
else:
    print("No optimal solution found.")

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 24 columns and 142 nonzeros
Model fingerprint: 0x5dd4a83c
Variable types: 19 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+06]
  Objective range  [1e+01, 6e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 2e+03]
Found heuristic solution: objective 0.0000000
Presolve removed 24 rows and 5 columns
Presolve time: 0.00s
Presolved: 21 rows, 19 columns, 83 nonzeros
Variable types: 19 continuous, 0 integer (0 binary)

Root relaxation: objective -2.341000e+03, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    -2341.0

##3.4

In [11]:
alpha = 0.85    # CVaR 신뢰수준
lam   = 100     # λ

m = gp.Model("CVaR_SLP")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
eta = m.addVar(lb=-GRB.INFINITY, name="eta")
v   = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam * eta + (lam/(1-alpha)) * gp.quicksum(p[s]*v[s] for s in scenarios)

m.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         eta
        +         v[s]
        >= 0,
        name=f"cvar_s{s}"
    )

# 결과
m.Params.OutputFlag = 1
m.optimize()

print(f"\n▶ CVaR SLP (α={alpha}, λ={lam}) 최적값 = {m.objVal:.2f}\n")
print("x =", [x[i].x for i in range(4)])
print("η =", eta.x)
print("v =", [v[s].x for s in scenarios])
print()

for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)])

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 25 columns and 147 nonzeros
Model fingerprint: 0xc7f0c5a6
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 3 columns
Presolve time: 0.00s
Presolved: 26 rows, 22 columns, 125 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
      27   -1.2625000e+05   0.000000e+00   0.000000e+00      0s

Solved in 27 iterations and 0.00 seconds (0.00 work units)
Optimal objective -1.262500000e+05

▶ CVaR SLP (α=0.85, λ=100) 최적값 = -126250.00

x = [129.99999999999997, 389.99999999999994, 239.99999999999997, 169.99999999999997]
η = -1249

In [12]:
alpha = 0.95    # CVaR 신뢰수준
lam   = 100     # λ

m = gp.Model("CVaR_SLP")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
eta = m.addVar(lb=-GRB.INFINITY, name="eta")
v   = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam * eta + (lam/(1-alpha)) * gp.quicksum(p[s]*v[s] for s in scenarios)

m.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         eta
        +         v[s]
        >= 0,
        name=f"cvar_s{s}"
    )

# 결과
m.Params.OutputFlag = 1
m.optimize()

print(f"\n▶ CVaR SLP (α={alpha}, λ={lam}) 최적값 = {m.objVal:.2f}\n")
print("x =", [x[i].x for i in range(4)])
print("η =", eta.x)
print("v =", [v[s].x for s in scenarios])
print()

for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)])

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 25 columns and 147 nonzeros
Model fingerprint: 0xa3288d7c
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 5 columns
Presolve time: 0.00s
Presolved: 26 rows, 20 columns, 123 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
      24   -1.2625000e+05   0.000000e+00   0.000000e+00      0s

Solved in 24 iterations and 0.00 seconds (0.00 work units)
Optimal objective -1.262500000e+05

▶ CVaR SLP (α=0.95, λ=100) 최적값 = -126250.00

x = [130.0, 390.0, 240.0, 170.0]
η = -1250.0000000000002
v = [0.0, 0.0, 0.0, 0.0, 0.0]

Scenar

In [13]:
alpha = 0.95    # CVaR 신뢰수준
lam   = 0     # λ

m = gp.Model("CVaR_SLP")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
eta = m.addVar(lb=-GRB.INFINITY, name="eta")
v   = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam * eta + (lam/(1-alpha)) * gp.quicksum(p[s]*v[s] for s in scenarios)

m.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         eta
        +         v[s]
        >= 0,
        name=f"cvar_s{s}"
    )

# 결과
m.Params.OutputFlag = 1
m.optimize()

print(f"\n▶ CVaR SLP (α={alpha}, λ={lam}) 최적값 = {m.objVal:.2f}\n")
print("x =", [x[i].x for i in range(4)])
print("η =", eta.x)
print("v =", [v[s].x for s in scenarios])
print()

for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)])

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 25 columns and 147 nonzeros
Model fingerprint: 0x59f8df34
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 24 rows and 6 columns
Presolve time: 0.00s
Presolved: 21 rows, 19 columns, 83 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.6988095e+03   4.092262e+01   0.000000e+00      0s
       9   -2.3410000e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.341000000e+03

▶ CVaR SLP (α=0.95, λ=0) 최적값 = -2341.00

x = [236.4, 690.0, 432.0, 318.0]
η = 180.0
v = [0.0, 0.0, 0.0, 0.0, 0.0]

Scenario 1: y = [15.0, 10.

##3.5

In [14]:
eta = 0       # η = 0
lam = 1000    # λ = 1000

m = gp.Model("EE_SLP")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
v = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam * gp.quicksum(p[s] * v[s] for s in scenarios)

m.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         v[s]
        >= -eta,
        name=f"EE_s{s}"
    )

m.Params.OutputFlag = 1
m.optimize()

# 결과
print(f"\n▶ Expected‐Excess SLP (η={eta}, λ={lam}) 최적값 = {m.objVal:.2f}\n")

print("x =", [x[i].x for i in range(4)])
for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)], f", v = {v[s].x:.2f}")


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 24 columns and 142 nonzeros
Model fingerprint: 0xcd65b2b0
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 26 rows, 24 columns, 123 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.6988095e+03   1.255952e+03   0.000000e+00      0s
      12   -2.2388776e+03   0.000000e+00   0.000000e+00      0s

Solved in 12 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.238877551e+03

▶ Expected‐Excess SLP (η=0, λ=1000) 최적값 = -2238.88

x = [234.48979591836738, 690.0, 429.7959183673469, 312.8571428571429]
Scenario 1: y = [

In [15]:
eta = -2341       # η = -2341
lam = 1000    # λ = 1000

m = gp.Model("EE_SLP")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
v = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam * gp.quicksum(p[s] * v[s] for s in scenarios)

m.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         v[s]
        >= -eta,
        name=f"EE_s{s}"
    )

m.Params.OutputFlag = 1
m.optimize()

# 결과
print(f"\n▶ Expected‐Excess SLP (η={eta}, λ={lam}) 최적값 = {m.objVal:.2f}\n")

print("x =", [x[i].x for i in range(4)])
for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)], f", v = {v[s].x:.2f}")


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 24 columns and 142 nonzeros
Model fingerprint: 0x53a6d64a
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 0 columns
Presolve time: 0.00s
Presolved: 26 rows, 24 columns, 123 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.6988095e+03   1.255952e+03   0.000000e+00      0s
      25    2.6561335e+05   0.000000e+00   0.000000e+00      0s

Solved in 25 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.656133533e+05

▶ Expected‐Excess SLP (η=-2341, λ=1000) 최적값 = 265613.35

x = [226.06666666666672, 671.9866666666667, 415.9200000000001, 298.4933333333334]


In [16]:
eta = 0       # η = 0
lam = 0    # λ = 0

m = gp.Model("EE_SLP")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
v = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam * gp.quicksum(p[s] * v[s] for s in scenarios)

m.setObjective(cost_term - exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         v[s]
        >= -eta,
        name=f"EE_s{s}"
    )

m.Params.OutputFlag = 1
m.optimize()

# 결과
print(f"\n▶ Expected‐Excess SLP (η={eta}, λ={lam}) 최적값 = {m.objVal:.2f}\n")

print("x =", [x[i].x for i in range(4)])
for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)], f", v = {v[s].x:.2f}")


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 24 columns and 142 nonzeros
Model fingerprint: 0x672faa78
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 24 rows and 5 columns
Presolve time: 0.00s
Presolved: 21 rows, 19 columns, 83 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.6988095e+03   4.092262e+01   0.000000e+00      0s
       9   -2.3410000e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.341000000e+03

▶ Expected‐Excess SLP (η=0, λ=0) 최적값 = -2341.00

x = [236.4, 690.0, 432.0, 318.0]
Scenario 1: y = [15.0, 10.0, 5.0] , v = 180.00
Scenario 2: 

##3.6

In [17]:
alpha = 0.9    #  신뢰수준
M     = 1e6

m = gp.Model("JointChance_abc_PPP")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
z = m.addVars(scenarios, vtype=GRB.BINARY, name="z")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
m.setObjective(cost_term - exp_revenue, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

for s in scenarios:
    da, db, dc = demand[s]
    m.addConstr(-y[s,0] + M*z[s] >= -da)
    m.addConstr(-y[s,1] + M*z[s] >= -db)
    m.addConstr(-y[s,2] + M*z[s] >= -dc)

m.addConstr(
    gp.quicksum(p[s]*z[s] for s in scenarios) <= 1 - alpha,
    "joint_chance"
)

m.Params.OutputFlag = 1
m.optimize()

# 결과
print(f"\n▶ Joint‐Chance SLP (α={alpha}) 최적값 = {m.objVal:.2f}\n")

print("x =", [x[i].x for i in range(4)])
for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)], ", z =", int(z[s].x))


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 41 rows, 24 columns and 122 nonzeros
Model fingerprint: 0x961c33b0
Variable types: 19 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e-01, 1e+06]
  Objective range  [1e+01, 6e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 2e+03]
Found heuristic solution: objective 0.0000000
Presolve removed 14 rows and 4 columns
Presolve time: 0.00s
Presolved: 27 rows, 20 columns, 95 nonzeros
Variable types: 19 continuous, 1 integer (1 binary)

Root relaxation: objective -2.875057e+03, 19 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 -2875.0575    0    1    0

In [18]:
alpha = 0.8    #  신뢰수준
M     = 1e6

m = gp.Model("JointChance_abc_PPP")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
z = m.addVars(scenarios, vtype=GRB.BINARY, name="z")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
m.setObjective(cost_term - exp_revenue, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

for s in scenarios:
    da, db, dc = demand[s]
    m.addConstr(-y[s,0] + M*z[s] >= -da)
    m.addConstr(-y[s,1] + M*z[s] >= -db)
    m.addConstr(-y[s,2] + M*z[s] >= -dc)

m.addConstr(
    gp.quicksum(p[s]*z[s] for s in scenarios) <= 1 - alpha,
    "joint_chance"
)

m.Params.OutputFlag = 1
m.optimize()

# 결과
print(f"\n▶ Joint‐Chance SLP (α={alpha}) 최적값 = {m.objVal:.2f}\n")

print("x =", [x[i].x for i in range(4)])
for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)], ", z =", int(z[s].x))


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 41 rows, 24 columns and 122 nonzeros
Model fingerprint: 0xfdcb70f9
Variable types: 19 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e-01, 1e+06]
  Objective range  [1e+01, 6e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-01, 2e+03]
Found heuristic solution: objective 0.0000000
Presolve removed 12 rows and 4 columns
Presolve time: 0.00s
Presolved: 29 rows, 20 columns, 99 nonzeros
Variable types: 19 continuous, 1 integer (1 binary)

Root relaxation: objective -3.174000e+03, 13 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    -3174.

##3.7

In [27]:
alpha = 0.85
epsilon1 = 0.15
epsilon2 = 0.85
lam   = 1/epsilon2

m = gp.Model("Quantile_Deviation")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
eta = m.addVar(lb=-GRB.INFINITY, name="eta")
v   = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam* epsilon1* eta + (lam*( epsilon1+ epsilon2)) * gp.quicksum(p[s]*v[s] for s in scenarios)

m.setObjective((1-lam*epsilon1)*cost_term - (1-lam*epsilon1)*exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         eta
        +         v[s]
        >= 0,
        name=f"QD{s}"
    )

# 결과
m.Params.OutputFlag = 1
m.optimize()

print(f"\n▶ Quantile Deviation (α={alpha}, ε1={epsilon1},ε2={epsilon2},λ={lam}) 최적값 = {m.objVal:.2f}\n")
print("x =", [x[i].x for i in range(4)])
print("η =", eta.x)
print("v =", [v[s].x for s in scenarios])
print()

for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)])

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 25 columns and 147 nonzeros
Model fingerprint: 0x6db2260f
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e-01, 5e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 3 columns
Presolve time: 0.00s
Presolved: 26 rows, 22 columns, 125 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
      14   -1.9814118e+03   0.000000e+00   0.000000e+00      0s

Solved in 14 iterations and 0.01 seconds (0.00 work units)
Optimal objective -1.981411765e+03

▶ Quantile Deviation (α=0.85, ε1=0.15,ε2=0.85,λ=1.1764705882352942) 최적값 = -1981.41

x = [236.4, 690.0, 432.0, 318.0]
η = -1270.0
v = [1450.

In [25]:
alpha = 0.95
epsilon1 = 0.05
epsilon2 = 0.95
lam   = 1/epsilon2

m = gp.Model("Quantile_Deviation")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
eta = m.addVar(lb=-GRB.INFINITY, name="eta")
v   = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam* epsilon1* eta + (lam*( epsilon1+ epsilon2)) * gp.quicksum(p[s]*v[s] for s in scenarios)

m.setObjective((1-lam*epsilon1)*cost_term - (1-lam*epsilon1)*exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         eta
        +         v[s]
        >= 0,
        name=f"QD{s}"
    )

# 결과
m.Params.OutputFlag = 1
m.optimize()

print(f"\n▶ Quantile Deviation (α={alpha}, ε1={epsilon1},ε2={epsilon2},λ={lam}) 최적값 = {m.objVal:.2f}\n")
print("x =", [x[i].x for i in range(4)])
print("η =", eta.x)
print("v =", [v[s].x for s in scenarios])
print()

for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)])

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 25 columns and 147 nonzeros
Model fingerprint: 0x5cb16cdb
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [5e-02, 5e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 19 rows and 5 columns
Presolve time: 0.00s
Presolved: 26 rows, 20 columns, 123 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
      12   -2.2149500e+03   0.000000e+00   0.000000e+00      0s

Solved in 12 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.214950000e+03

▶ Quantile Deviation (α=0.95, ε1=0.05,ε2=0.95,λ=1) 최적값 = -2214.95

x = [236.4, 690.0, 432.0, 318.0]
η = 180.0
v = [0.0, 0.0, 0.0, 0.0, 0.0]

In [21]:
# 람다가 0일때
alpha = 0.85
epsilon1 = 0.15
epsilon2 = 0.85
lam   = 0

m = gp.Model("Quantile_Deviation")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
eta = m.addVar(lb=-GRB.INFINITY, name="eta")
v   = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam* epsilon1* eta + (lam*( epsilon1+ epsilon2)) * gp.quicksum(p[s]*v[s] for s in scenarios)

m.setObjective((1-lam*epsilon1)*cost_term - (1-lam*epsilon1)*exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         eta
        +         v[s]
        >= 0,
        name=f"QD{s}"
    )

# 결과
m.Params.OutputFlag = 1
m.optimize()

print(f"\n▶ Quantile Deviation (α={alpha}, ε1={epsilon1},ε2={epsilon2},λ={lam}) 최적값 = {m.objVal:.2f}\n")
print("x =", [x[i].x for i in range(4)])
print("η =", eta.x)
print("v =", [v[s].x for s in scenarios])
print()

for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)])

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 25 columns and 147 nonzeros
Model fingerprint: 0x59f8df34
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 24 rows and 6 columns
Presolve time: 0.00s
Presolved: 21 rows, 19 columns, 83 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.6988095e+03   4.092262e+01   0.000000e+00      0s
       9   -2.3410000e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.341000000e+03

▶ Quantile Deviation (α=0.85, ε1=0.15,ε2=0.85,λ=0) 최적값 = -2341.00

x = [236.4, 690.0, 432.0, 318.0]
η = 180.0
v = [0.0, 0.0, 0.0, 0.0, 0.0]



In [22]:
# 람다가 0일때
alpha = 0.95
epsilon1 = 0.05
epsilon2 = 0.95
lam   = 0

m = gp.Model("Quantile_Deviation")

# 변수
x = m.addVars(4, lb=0.0, name="x")
y = m.addVars(scenarios, 3, lb=0.0, name="y")
eta = m.addVar(lb=-GRB.INFINITY, name="eta")
v   = m.addVars(scenarios, lb=0.0, name="v")

# 목적식
cost_term = gp.quicksum(cost_coef[i] * x[i] for i in range(4))
exp_revenue = gp.quicksum(
    p[s] * (rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2])
    for s in scenarios
)
risk_term = lam* epsilon1* eta + (lam*( epsilon1+ epsilon2)) * gp.quicksum(p[s]*v[s] for s in scenarios)

m.setObjective((1-lam*epsilon1)*cost_term - (1-lam*epsilon1)*exp_revenue + risk_term, GRB.MINIMIZE)

# 제약식
m.addConstr(x[0] <= 300)
m.addConstr(x[1] <= 700)
m.addConstr(x[2] <= 600)
m.addConstr(x[3] <= 500)
m.addConstr(x[1] + x[2] + x[3] <= 1600)

for s in scenarios:
    da, db, dc = demand[s]

    m.addConstr(x[0] >=   6*y[s,0] +  8*y[s,1] + 10*y[s,2])
    m.addConstr(x[1] >=  20*y[s,0] + 25*y[s,1] + 28*y[s,2])
    m.addConstr(x[2] >=  12*y[s,0] + 15*y[s,1] + 18*y[s,2])
    m.addConstr(x[3] >=   8*y[s,0] + 10*y[s,1] + 14*y[s,2])

    m.addConstr(y[s,0] <= da)
    m.addConstr(y[s,1] <= db)
    m.addConstr(y[s,2] <= dc)

    rev_s = rev_coef[0]*y[s,0] + rev_coef[1]*y[s,1] + rev_coef[2]*y[s,2]
    m.addConstr(
        - cost_term
        +         rev_s
        +         eta
        +         v[s]
        >= 0,
        name=f"QD{s}"
    )

# 결과
m.Params.OutputFlag = 1
m.optimize()

print(f"\n▶ Quantile Deviation (α={alpha}, ε1={epsilon1},ε2={epsilon2},λ={lam}) 최적값 = {m.objVal:.2f}\n")
print("x =", [x[i].x for i in range(4)])
print("η =", eta.x)
print("v =", [v[s].x for s in scenarios])
print()

for s in scenarios:
    print(f"Scenario {s}: y =", [y[s,k].x for k in range(3)])

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45 rows, 25 columns and 147 nonzeros
Model fingerprint: 0x59f8df34
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+01, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 2e+03]
Presolve removed 24 rows and 6 columns
Presolve time: 0.00s
Presolved: 21 rows, 19 columns, 83 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.6988095e+03   4.092262e+01   0.000000e+00      0s
       9   -2.3410000e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.00 seconds (0.00 work units)
Optimal objective -2.341000000e+03

▶ Quantile Deviation (α=0.95, ε1=0.05,ε2=0.95,λ=0) 최적값 = -2341.00

x = [236.4, 690.0, 432.0, 318.0]
η = 180.0
v = [0.0, 0.0, 0.0, 0.0, 0.0]

